# Alpaca Eval Results Visualization

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
import glob
import numpy as np

# Set plot style
sns.set_theme(style="whitegrid")

## Helper Functions

In [ ]:
def process_df_names(df):
    df['alpha'] = df['model_name'].str.extract(r"-a([\d\.]+)").astype(float)
    df['beta'] = df['model_name'].str.extract(r"-b([\d\.]+)").astype(float)
    df['L'] = df['model_name'].str.extract(r"-L(\d+)").astype(float) # float because NaN issues if missing, convert later
    df['epochs'] = df['model_name'].str.extract(r"-e(\d+)").astype(float)
    df['mt'] = df['model_name'].str.extract(r"_mt(\d+)").astype(float)
    df['clean_name'] = df['model_name'].str.split("-e").str[0] 
    df['model'] = df['model_name'].str.split("-a").str[0]
    df['model_type'] = 'pepo'
    dpo_mask = (df['L'] == 1) & (df['alpha'] == 0.0)
    df.loc[dpo_mask, 'model_type'] = 'dpo'
    initial_mask = (df['epochs'] == 0)
    df.loc[initial_mask, 'model_type'] = 'initial'
    return df

def simple_name(str, include_epochs=True):
    if include_epochs:
        if str == "initial":
            return "Initial"
        elif str == "dpo":
            return "DPO"
        else:
            return f"PEPO (L={int(str.split("-")[1])})"
    else:
        if str == "initial":
            return "Initial"
        elif str == "dpo":
            return "DPO"
        else:
            return f"PEPO (L={int(str.split("-")[1])})"


def load_experiments(base_dir="eval_outputs/alpaca_eval"):
    """
    Loads leaderboard.csv from subdirectories using vectorized operations.
    Filters: 
    - keeps 'default' and folders containing 'mt1024'
    - models must contain 'mt1024'
    - removes '-pepo' models
    """
    experiments = {}
    
    # Walk through the directory
    for root, dirs, files in os.walk(base_dir):
        if "leaderboard.csv" in files:
            folder_name = os.path.basename(root)
            
            # Filter condition
            if folder_name == "default" or "mt1024" in folder_name:
                file_path = os.path.join(root, "leaderboard.csv")
                try:
                    df = pd.read_csv(file_path, index_col=0)
                    
                    # 1. Set model_name from index
                    df['model_name'] = df.index.astype(str)
                    
                    # 2. Filter rows
                    # Remove '-pepo' AND Keep models with 'mt1024'
                    mask = (~df['model_name'].str.contains("-pepo")) & (df['model_name'].str.contains("mt1024"))
                    df = df[mask].copy()
                    
                    if not df.empty:
                        df = process_df_names(df)
                        df.set_index('model_name', inplace=True)
                        
                        experiments[folder_name] = df
                        print(f"Loaded {folder_name} with {len(experiments[folder_name])} rows.")
                    else:
                        print(f"Skipping {folder_name}: No models left after filtering.")
                        
                except Exception as e:
                    print(f"Error loading {folder_name}: {e}")
                    
    return experiments

In [ ]:
experiments = load_experiments()

In [ ]:
experiments['default'].head()

In [ ]:
names = ['Llama-3.1-Tulu-3-8B-SFT','zephyr-7b-sft-full', 'mistral-7b-sft-beta', 'Yi'] #, 'Llama-3.2-3B-Instruct']

for key in experiments:
    tmp = experiments[key].sort_values(by='win_rate', ascending=False)
    # if none of the names are contained in any of the clean names, skip
    if not tmp['clean_name'].str.contains('|'.join(names)).any():
        continue

    for idx in range(len(names)):
        table = tmp[tmp['clean_name'].str.contains(names[idx])]
        if len(table) == 0:
            continue
        print(f"{names[idx]}   vs   {key if key != 'default' else 'gpt4'}")
        display(table[['win_rate','standard_error']])

# gpt4 vs our models compared on alpaca eval

In [ ]:
df = experiments['default'].sort_values(by='win_rate', ascending=False)
df['model'].unique()

In [ ]:
def plot_win_rates(alpaca_eval_df,ax=None,epochs=None):
    if ax is None:
        fig, ax = plt.subplots()
    # plot win rate by epoch and area with lighter color of std
    alpaca_eval_df = alpaca_eval_df.sort_values(by='epochs')
    # filter epochs
    if epochs is not None:
        alpaca_eval_df = alpaca_eval_df[alpaca_eval_df['epochs'].isin(epochs)]
    display(alpaca_eval_df)
    label = alpaca_eval_df.index[1]
    ax.plot(alpaca_eval_df['epochs'],alpaca_eval_df['win_rate'],label=label, marker='o')
    ax.fill_between(
        alpaca_eval_df['epochs'], 
        alpaca_eval_df['win_rate'] - alpaca_eval_df['standard_error'], 
        alpaca_eval_df['win_rate'] + alpaca_eval_df['standard_error'], 
        alpha=0.2,
        # label=label
    )

# models = ['zephyr-7b-sft-full', 'Yi-34B-Chat']
# models = ['mistral-7b-sft-beta']
# models = ['Llama-3.1-Tulu-3-8B-SFT']
models = ['zephyr-7b-sft-full']
# epochs = [0,1,2,3, 6, 11]
epochs = None

for model in df['model'].unique():
    if model not in models:
        continue
    fig,ax = plt.subplots()
    tmp = df[df['model'] == model]
    # display(tmp)
    for name in tmp['clean_name'].unique():
        plot_win_rates(tmp[tmp['clean_name'] == name], ax=ax,epochs=epochs)
    ax.legend()
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Win Rate')
    ax.set_title(f"Alpaca Eval of {model} (agains gpt4)")
    plt.show()
    